# Education Data Pipeline — Interactive Walkthrough

This notebook walks through the full pipeline end-to-end using Nevada test scores as a concrete example.

**Sections**
1. Setup & imports
2. Create folder structure
3. Run a single downloader (Nevada test scores)
4. Inspect the raw downloaded file
5. Run the cleaner
6. Inspect cleaned output

---
## 1. Setup & Imports

Make sure you have installed all dependencies first:

```bash
pip install -r requirements.txt
```

We add the project root to `sys.path` so all local modules are importable from this notebook.

In [ ]:
import sys
from pathlib import Path

# Ensure the notebook can find the project root regardless of where it is launched from
PROJECT_ROOT = Path(".").resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

import pandas as pd
import config

print(f"States configured : {config.STATES}")
print(f"Categories        : {config.CATEGORIES}")

---
## 2. Create Folder Structure

`create_dir_structure()` creates all `raw/` and `cleaned/` subdirectories
for both states and all categories. It is idempotent — safe to run multiple times.

In [ ]:
from scripts.utils.file_utils import create_dir_structure

create_dir_structure()

# Verify a few key directories exist
for state in config.STATES:
    for category in config.CATEGORIES:
        raw = config.raw_dir(state, category)
        cleaned = config.cleaned_dir(state, category)
        assert raw.exists(),     f"Missing: {raw}"
        assert cleaned.exists(), f"Missing: {cleaned}"

print("✓ All directories created successfully.")

---
## 3. Run the Nevada Test Scores Downloader

We instantiate `NevadaTestScoresDownloader` and call `download_all()`.

The downloader will:
- Attempt to fetch each URL in `config.URLS['nevada']['test_scores']`
- Retry up to 3 times on transient failures
- Save files to `data/nevada/raw/test_scores/`
- Log each attempt to `logs/download_manifest.csv`

> **Note**: Some URLs point to HTML listing pages rather than direct file downloads.
> Expect those to save as `.html` files — the downloader correctly detects the content type.
> Extend `download_all()` with BeautifulSoup scraping to extract direct file links.

In [ ]:
from scripts.downloaders.nevada.nv_test_scores import NevadaTestScoresDownloader

downloader = NevadaTestScoresDownloader()
downloader.download_all()

raw_dir = config.raw_dir("nevada", "test_scores")
downloaded_files = list(raw_dir.iterdir())
print(f"\nFiles in {raw_dir}:")
for f in downloaded_files:
    print(f"  {f.name}  ({f.stat().st_size / 1024:.1f} KB)")

---
## 4. Inspect a Raw Downloaded File

Pick the first downloaded file and load it into a DataFrame to see what the raw data looks like before cleaning.

In [ ]:
raw_dir = config.raw_dir("nevada", "test_scores")
supported_exts = {".xlsx", ".xls", ".csv"}

raw_files = [f for f in raw_dir.iterdir() if f.suffix.lower() in supported_exts]

if not raw_files:
    print("No tabular files found yet. Run section 3 first, or place a sample file in the raw directory.")
else:
    sample_file = raw_files[0]
    print(f"Inspecting: {sample_file.name}")

    if sample_file.suffix.lower() in {".xlsx", ".xls"}:
        df_raw = pd.read_excel(sample_file, dtype=str)
    else:
        df_raw = pd.read_csv(sample_file, dtype=str, encoding="utf-8", encoding_errors="replace")

    print(f"Shape: {df_raw.shape}")
    print(f"Columns: {list(df_raw.columns)}")
    df_raw.head()

---
## 5. Run the Test Scores Cleaner

The cleaner reads every file in `data/nevada/raw/test_scores/`, applies
standardisation, and writes cleaned CSVs to `data/nevada/cleaned/test_scores/`.

In [ ]:
from scripts.cleaners.clean_test_scores import clean

clean(state="nevada")

cleaned_dir = config.cleaned_dir("nevada", "test_scores")
cleaned_files = list(cleaned_dir.glob("*_cleaned.csv"))
print(f"\nCleaned files in {cleaned_dir}:")
for f in cleaned_files:
    print(f"  {f.name}")

---
## 6. Inspect Cleaned Output

Load the first cleaned CSV and verify the column naming, metadata columns, and overall shape.

In [ ]:
cleaned_dir = config.cleaned_dir("nevada", "test_scores")
cleaned_files = list(cleaned_dir.glob("*_cleaned.csv"))

if not cleaned_files:
    print("No cleaned files found. Run section 5 first.")
else:
    df_clean = pd.read_csv(cleaned_files[0], dtype=str, encoding="utf-8")

    print(f"File  : {cleaned_files[0].name}")
    print(f"Shape : {df_clean.shape}")
    print(f"\nColumn names:")
    for col in df_clean.columns:
        print(f"  {col}")

    print("\n--- Sample rows ---")
    df_clean.head(10)

---
## Bonus: Check the Download Manifest

Every download attempt is logged to `logs/download_manifest.csv`.
Load it here to audit what was fetched.

In [ ]:
manifest_path = config.MANIFEST_CSV

if manifest_path.exists():
    df_manifest = pd.read_csv(manifest_path)
    print(f"Manifest rows: {len(df_manifest)}")
    df_manifest
else:
    print(f"Manifest not found at {manifest_path}. Run the downloader first.")